# Job Scraping Analysis Notebook
**Name**: Mayenmein Terence Sama Aloah Jr<br>
**Date**: October 2025<br>
**Project**: SkillHub Job Data Collection
## Introduction
This notebook demonstrates the functionality of the JobScraper class for collecting job posting data from the Found.dev API and storing it directly in PostgreSQL database. The implementation focuses on batch processing, database efficiency, and progress tracking.

## 1. Import and Setup
Let's start by importing the necessary modules and setting up our environment.

In [1]:
import sys
import os
import pandas as pd
from datetime import datetime
from pathlib import Path
import psycopg2
# Add the src directory to the path to import our custom module
sys.path.append('..')

# Import the JobScraper class
from src.scraping.scrape_jobs import JobScraper
from src.scraping.scrape_cam import JobDataAPIStreamingScraper
from src.database.create_database import DatabaseCreator

print("✅ Imports completed successfully!")
print(f"📅 Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Imports completed successfully!
📅 Analysis date: 2026-02-23 13:00:19


## 3. Test Database Connection and Schema
Verify that we can connect to the database and check the table structure.

In [2]:
database_manager = DatabaseCreator()
database_manager.run()

Connecting to PostgreSQL at localhost:5432...
Connected to PostgreSQL as postgres
Database 'data_science_job_market_db' already exists
Successfully connected to 'data_science_job_market_db' with regular user credentials


In [3]:
datajobs_scraper = JobDataAPIStreamingScraper()

 Tables created successfully from c:\Users\MARIE\Desktop\TERENCE\PERSONAL PROJECTS\skillhub\notebooks\..\sql\00 schema\create_raw_tables_datajobs.sql
 Existing tables: companies, jobs, job_skills_detail, job_listings_with_company, datajob_to_types, jobs_cleaned, datajob_types, datajob_locations, datajobslocations, datajobscountries, datajobscompanies, datajobs


In [6]:
total = datajobs_scraper.scrape_by_country('NG')
datajobs_scraper.get_database_stats()

Streaming jobs from JobDataAPI for country: NG


Processing jobs: 100jobs [00:06, 14.41jobs/s, total=100]



Streaming complete! Total jobs processed: 100

DATABASE STATISTICS
Total jobs: 200
Total companies: 80
Total locations: 51


{'jobs': 200, 'companies': 80, 'locations': 51}

## 2. Initialize the Job Scraper
Create an instance of the JobScraper class with custom configuration.

In [ ]:
scraper = JobScraper()

print(f"Scraper initialized successfully!")
print(f"API endpoint: {scraper.BASE_URL}")
print(f"Database: {scraper.db_config['dbname']} on {scraper.db_config['host']}")


## 4. Test Single Page Fetch
Before running full batch scraping, let's test fetching a single page to understand the data structure.

In [ ]:
def test_single_fetch():
    """Test fetching a single page of job data"""
    print("🔍 Testing single page fetch...")
    
    try:
        # Fetch first page
        data = scraper.fetch_jobs(page=1, skill="Data Science", ai=True)
        jobs = data.get("jobs", [])
        
        print(f"Jobs found on page 1: {len(jobs)}")
        
        if jobs:
            # Process the jobs
            companies, jobs, skill_details = scraper.process_job_data(jobs[:2])  # Process first 2 jobs as sample
            print(f"Processed jobs sample: {len(jobs)}")
            print(f"Processed companies sample: {len(companies)}")
            print(f"Processed skill details sample: {len(skill_details)}")
            
            # Display sample data
            if companies:
                company_df = pd.DataFrame(companies)
            if skill_details:
                skill_df = pd.DataFrame(skill_details)
            if jobs:
                job_df = pd.DataFrame(jobs)
            if not job_df.empty and not company_df.empty and not skill_df.empty:
                display(company_df.head(), skill_df.head(), job_df.head())
        return len(jobs)
        
    except Exception as e:
        print(f"Error during test fetch: {e}")
        return 0

# Run the test
jobs_count = test_single_fetch()
print(f"\n✅ Single page test completed. Found {jobs_count} jobs.")

## 5. Run Small Batch Scraping
Now let's run a small batch scraping operation to demonstrate the functionality with database storage.

In [ ]:
def run_small_batch_scraping():
    """Run scraping with a small batch size for demonstration"""
    print("Starting small batch scraping...")
    print("Jobs will be saved directly to PostgreSQL database")
    
    # Run with small batch size for quick demonstration
    total_jobs = scraper.scrape_in_batches(
        skill="Data Science",
        pages_per_batch=3,
        ai=True,
        delay=1,
        max_batches=2
    )
    
    print(f"\nSmall batch scraping completed!")
    print(f"Total jobs collected: {total_jobs}")
    
    return total_jobs

# Execute small batch scraping
small_batch_total = run_small_batch_scraping()

## 7. Advanced Database Analysis
Perform deeper analysis using SQL aggregations.

In [ ]:
def advanced_database_analysis():
    """Perform advanced analysis using SQL"""
    print("🔬 Performing advanced database analysis...")
    
    try:
        with psycopg2.connect(**scraper.db_config) as conn:
            # Skills analysis (using PostgreSQL array functions)
            skills_analysis = pd.read_sql("""
                SELECT 
                    unnest(skills) as skill,
                    COUNT(*) as frequency
                FROM raw_jobs
                WHERE skills IS NOT NULL
                GROUP BY skill
                ORDER BY frequency DESC
                LIMIT 15
            """, conn)
            
            print("\n🔥 TOP IN-DEMAND SKILLS:")
            print("-" * 40)
            for _, row in skills_analysis.iterrows():
                print(f"  {row['skill']}: {row['frequency']} jobs")
            
            # Job type distribution
            job_types = pd.read_sql("""
                SELECT 
                    type,
                    COUNT(*) as count,
                    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) as percentage
                FROM raw_jobs
                WHERE type IS NOT NULL
                GROUP BY type
                ORDER BY count DESC
            """, conn)
            
            print("\n📋 JOB TYPE DISTRIBUTION:")
            print("-" * 40)
            for _, row in job_types.iterrows():
                print(f"  {row['type']}: {row['count']} ({row['percentage']}%)")
            
            # AI vs Non-AI jobs
            ai_jobs = pd.read_sql("""
                SELECT 
                    ai,
                    COUNT(*) as count
                FROM raw_jobs
                GROUP BY ai
            """, conn)
            
            print("\n🤖 AI-RELATED JOBS:")
            print("-" * 40)
            ai_count = ai_jobs[ai_jobs['ai'] == True]['count'].values[0] if True in ai_jobs['ai'].values else 0
            non_ai_count = ai_jobs[ai_jobs['ai'] == False]['count'].values[0] if False in ai_jobs['ai'].values else 0
            print(f"  AI-related: {ai_count} jobs")
            print(f"  Non-AI: {non_ai_count} jobs")
            print(f"  AI percentage: {100 * ai_count / (ai_count + non_ai_count):.1f}%")
            
    except Exception as e:
        print(f"❌ Advanced analysis failed: {e}")

# Run advanced analysis
advanced_database_analysis()


## 8. Data Quality Check via Database
Perform data quality checks by querying the database.

In [ ]:
def query_database_for_analysis():
    """Query the database to analyze collected job data"""
    print("📈 Querying database for analysis...")
    
    try:
        with psycopg2.connect(**scraper.db_config) as conn:
            # 1. JOBS TABLE STATISTICS
            jobs_stats = pd.read_sql("""
                SELECT 
                    COUNT(*) as total_jobs,
                    COUNT(DISTINCT company_slug) as unique_companies,
                    COUNT(DISTINCT country) as unique_countries,
                    COUNT(DISTINCT city) as unique_cities,
                    SUM(CASE WHEN ai THEN 1 ELSE 0 END) as ai_jobs,
                    AVG(CASE WHEN salary_min > 0 THEN salary_min ELSE NULL END) as avg_min_salary,
                    AVG(CASE WHEN salary_max > 0 THEN salary_max ELSE NULL END) as avg_max_salary,
                    MIN(published) as oldest_job,
                    MAX(published) as newest_job
                FROM jobs
            """, conn)
            
            print("\n📊 JOBS TABLE STATISTICS")
            print("=" * 50)
            print(f"Total jobs: {jobs_stats['total_jobs'].values[0]:,}")
            print(f"Unique companies: {jobs_stats['unique_companies'].values[0]:,}")
            print(f"AI-related jobs: {jobs_stats['ai_jobs'].values[0]:,}")
            print(f"Unique countries: {jobs_stats['unique_countries'].values[0]:,}")
            print(f"Unique cities: {jobs_stats['unique_cities'].values[0]:,}")
            print(f"Avg salary range: ${jobs_stats['avg_min_salary'].values[0]:,.0f} - ${jobs_stats['avg_max_salary'].values[0]:,.0f}")
            print(f"Date range: {jobs_stats['oldest_job'].values[0]} to {jobs_stats['newest_job'].values[0]}")
            
            # 2. COMPANIES TABLE STATISTICS
            companies_stats = pd.read_sql("""
                SELECT 
                    COUNT(*) as total_companies,
                    COUNT(DISTINCT country) as company_countries,
                    AVG(linkedin_staff_count) as avg_staff_count,
                    SUM(jobs_count) as total_jobs_posted,
                    SUM(jobs_ai_count) as total_ai_jobs_posted,
                    AVG(EXTRACT(YEAR FROM AGE(CURRENT_DATE, TO_DATE(year_founded::text, 'YYYY')))) as avg_company_age
                FROM companies
                WHERE year_founded > 0
            """, conn)
            
            print("\n\n🏢 COMPANIES TABLE STATISTICS")
            print("=" * 50)
            print(f"Total companies: {companies_stats['total_companies'].values[0]:,}")
            print(f"Countries represented: {companies_stats['company_countries'].values[0]:,}")
            print(f"Avg employee count: {companies_stats['avg_staff_count'].values[0]:,.0f}")
            print(f"Total jobs posted: {companies_stats['total_jobs_posted'].values[0]:,}")
            print(f"Total AI jobs posted: {companies_stats['total_ai_jobs_posted'].values[0]:,}")
            print(f"Avg company age: {companies_stats['avg_company_age'].values[0]:.1f} years")
            
            # 3. JOB_SKILLS_DETAIL TABLE STATISTICS
            skills_stats = pd.read_sql("""
                SELECT 
                    COUNT(DISTINCT skill_name) as unique_skills,
                    COUNT(*) as total_skill_mentions,
                    skill_category,
                    COUNT(*) as category_count,
                    COUNT(DISTINCT skill_name) as unique_skills_in_category
                FROM job_skills_detail
                GROUP BY skill_category
                ORDER BY category_count DESC
            """, conn)
            
            print("\n\n🔧 SKILLS DETAIL STATISTICS")
            print("=" * 50)
            total_skills = skills_stats['total_skill_mentions'].sum() if not skills_stats.empty else 0
            unique_skills = skills_stats['unique_skills'].sum() if not skills_stats.empty else 0
            print(f"Total skill mentions: {total_skills:,}")
            print(f"Unique skills: {unique_skills:,}")
            print("\nSkills by category:")
            for _, row in skills_stats.iterrows():
                print(f"  └─ {row['skill_category']}: {row['category_count']:,} mentions ({row['unique_skills_in_category']} unique)")
            
            # 4. TOP SKILLS OVERALL
            top_skills = pd.read_sql("""
                SELECT 
                    skill_name,
                    COUNT(*) as mention_count,
                    COUNT(DISTINCT job_slug) as unique_jobs
                FROM job_skills_detail
                GROUP BY skill_name
                ORDER BY mention_count DESC
                LIMIT 15
            """, conn)
            
            print("\n\n🏆 TOP 15 SKILLS")
            print("=" * 50)
            for _, row in top_skills.iterrows():
                print(f"  {row['skill_name']}: {row['mention_count']} mentions ({row['unique_jobs']} jobs)")
            
            # 5. JOB TYPES DISTRIBUTION
            job_types = pd.read_sql("""
                SELECT 
                    unnest(types) as job_type,
                    COUNT(*) as count
                FROM jobs
                WHERE types IS NOT NULL
                GROUP BY job_type
                ORDER BY count DESC
            """, conn)
            
            print("\n\n📋 JOB TYPES DISTRIBUTION")
            print("=" * 50)
            for _, row in job_types.iterrows():
                print(f"  {row['job_type']}: {row['count']:,} jobs")
            
            # 6. TOP COMPANIES BY JOB COUNT
            top_companies = pd.read_sql("""
                SELECT 
                    c.name,
                    c.jobs_count,
                    c.jobs_ai_count,
                    ROUND(100.0 * c.jobs_ai_count / NULLIF(c.jobs_count, 0), 1) as ai_percentage
                FROM companies c
                WHERE c.jobs_count > 0
                ORDER BY c.jobs_count DESC
                LIMIT 10
            """, conn)
            
            print("\n\n🏅 TOP 10 COMPANIES BY JOB COUNT")
            print("=" * 60)
            for _, row in top_companies.iterrows():
                print(f"  {row['name']}: {row['jobs_count']} jobs ({row['jobs_ai_count']} AI, {row['ai_percentage']}%)")
            
            # 7. LOCATION BREAKDOWN
            locations = pd.read_sql("""
                SELECT 
                    country,
                    COUNT(*) as job_count,
                    COUNT(DISTINCT city) as cities_count
                FROM jobs
                WHERE country IS NOT NULL AND country != ''
                GROUP BY country
                ORDER BY job_count DESC
                LIMIT 10
            """, conn)
            
            print("\n\n🌍 TOP 10 COUNTRIES")
            print("=" * 50)
            for _, row in locations.iterrows():
                print(f"  {row['country']}: {row['job_count']} jobs ({row['cities_count']} cities)")
            
            # 8. SAMPLE JOBS WITH THEIR SKILLS
            sample_jobs = pd.read_sql("""
                SELECT 
                    j.title,
                    j.company_slug,
                    j.published,
                    array_to_string(j.skills, ', ') as skills_list
                FROM jobs j
                WHERE j.skills IS NOT NULL AND array_length(j.skills, 1) > 0
                ORDER BY j.published DESC
                LIMIT 5
            """, conn)
            
            print("\n\n📄 SAMPLE JOBS WITH SKILLS")
            print("=" * 70)
            for _, row in sample_jobs.iterrows():
                print(f"\n  Title: {row['title']}")
                print(f"  Company: {row['company_slug']}")
                print(f"  Skills: {row['skills_list']}")
            
            return {
                'jobs_stats': jobs_stats,
                'companies_stats': companies_stats,
                'skills_stats': skills_stats,
                'top_skills': top_skills,
                'job_types': job_types,
                'top_companies': top_companies,
                'locations': locations
            }
            
    except Exception as e:
        print(f"❌ Database query failed: {e}")
        return None

# Analyze the database
stats = query_database_for_analysis()

## 9. Full-Scale Scraping
For comprehensive data collection, run the full scraping process.

In [ ]:
def run_comprehensive_scraping():
    """Run comprehensive job scraping (optional - may take time)"""
    print("🔍 Starting comprehensive scraping...")
    print("⚠️  This may take 30-60 minutes depending on job volume")
    
    # You can adjust these parameters based on your needs
    total_jobs = scraper.scrape_in_batches(
        skill="Data Science",
        pages_per_batch=20,  # Larger batches for efficiency
        ai=True,
        delay=1,
        max_batches=None  # No limit, stops when no more jobs
    )
    
    print(f"\n🏁 Comprehensive scraping completed!")
    print(f"📈 Total jobs collected: {total_jobs}")
    
    return total_jobs

comprehensive_total = run_comprehensive_scraping()


In [ ]:
print("\n🎉 Notebook execution complete! All data is now stored in PostgreSQL database.")